# Mini-Project - Soccer Results Analytics

This notebook covers:
1. Historical data preparation with Spark
2. Current-season enrichment by web scraping
3. Creation of a clean prepared dataset for later streaming and machine learning tasks

The historical dataset is loaded, cleaned, enriched with derived columns, and saved in Parquet format.
The scraped current-season data is extracted separately and will later be used as an evaluation set for the ML part.

## Part 1 - Historical Data Preparation

In this part, we:
- load the historical CSV dataset into Spark
- inspect the schema and data quality
- convert the date column
- create derived columns required by the project
- handle missing values
- save the cleaned dataset in Parquet format

### 1.1 — Spark Session Setup

We initialize a Spark session connected to the Spark master running in Docker.
The `inferSchema` option will be used for initial loading, and we will then manually correct data types as needed.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("mini-project-prepare-data")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark version:", spark.version)
print("Master:", sc.master)
print("App name:", sc.appName)

Spark version: 3.5.3
Master: spark://spark-master:7077
App name: mini-project-prepare-data


### 1.2 — Loading the Historical Dataset

The CSV file `rim_championnat_results_2007-2025.csv` contains **2744 match records** from the Mauritanian football league.

Columns: `season`, `date` (dd.MM.yyyy format), `home_team`, `away_team`, `home_goals`, `away_goals`.

We use `inferSchema=True` for initial load, then verify and correct types manually.

In [3]:
file_path = "../data/rim_championnat_results_2007-2025.csv"

In [4]:
results_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

In [5]:
results_df.printSchema()
results_df.show(10, truncate=False)

root
 |-- season: string (nullable = true)
 |-- date: string (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: integer (nullable = true)
 |-- away_goals: integer (nullable = true)

+------------------------+----------+---------------+---------------+----------+----------+
|season                  |date      |home_team      |away_team      |home_goals|away_goals|
+------------------------+----------+---------------+---------------+----------+----------+
|Championnat D1 2007/2008|02.08.2008|ASAC Concorde  |Nasr Sebkha    |2         |0         |
|Championnat D1 2007/2008|02.08.2008|Police         |Tidjikja       |0         |1         |
|Championnat D1 2007/2008|01.08.2008|Ksar           |SNIM           |1         |0         |
|Championnat D1 2007/2008|01.08.2008|Mauritel Mobile|Armee          |0         |0         |
|Championnat D1 2007/2008|01.08.2008|Tevragh-Zeina  |FC Trarza      |3         |0         |
|Championnat D1 

In [6]:
print("Number of rows:", results_df.count())
print("Columns:", results_df.columns)

Number of rows: 2744
Columns: ['season', 'date', 'home_team', 'away_team', 'home_goals', 'away_goals']


In [7]:
results_df.describe().show()

+-------+--------------------+----------+--------------------+--------------------+------------------+------------------+
|summary|              season|      date|           home_team|           away_team|        home_goals|        away_goals|
+-------+--------------------+----------+--------------------+--------------------+------------------+------------------+
|  count|                2744|      2744|                2744|                2744|              2744|              2744|
|   mean|                NULL|      NULL|                NULL|                NULL|1.2463556851311954|1.0677842565597668|
| stddev|                NULL|      NULL|                NULL|                NULL|1.2974446346444488|1.1610070377473283|
|    min|Championnat D1 20...|    01.01.|       ASAC Concorde|       ASAC Concorde|                 0|                 0|
|    max|  Super D1 2017/2018|31.12.2024|Trarza Nadi Sporting|Trarza Nadi Sporting|                10|                10|
+-------+---------------

### 1.3 — Date Conversion

The `date` column is currently a string in `dd.MM.yyyy` format.
We convert it to a proper Spark `DateType` using `to_date()`.

**Note:** Some date strings are malformed (e.g., `01.01.` without year), which will produce `null` values after conversion. We will handle these in the cleaning step.

In [8]:
results_df = results_df.withColumn(
    "date",
    F.to_date(F.col("date"), "dd.MM.yyyy")
)

In [9]:
results_df.printSchema()
results_df.select("date").show(10, truncate=False)

root
 |-- season: string (nullable = true)
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: integer (nullable = true)
 |-- away_goals: integer (nullable = true)

+----------+
|date      |
+----------+
|2008-08-02|
|2008-08-02|
|2008-08-01|
|2008-08-01|
|2008-08-01|
|2008-07-29|
|2008-07-26|
|2008-07-26|
|2008-07-25|
|2008-07-25|
+----------+
only showing top 10 rows



### 1.4 — Derived Columns

We add the following required derived columns:

| Column | Formula |
|--------|--------|
| `total_goals` | `home_goals + away_goals` |
| `goal_difference` | `home_goals - away_goals` |
| `result` | `home_win` / `draw` / `away_win` based on score comparison |
| `match_year` | Extracted from the `date` column |

In [10]:
prepared_df = (
    results_df
    .withColumn("total_goals", F.col("home_goals") + F.col("away_goals"))
    .withColumn("goal_difference", F.col("home_goals") - F.col("away_goals"))
    .withColumn(
        "result",
        F.when(F.col("home_goals") > F.col("away_goals"), "home_win")
         .when(F.col("home_goals") < F.col("away_goals"), "away_win")
         .otherwise("draw")
    )
    .withColumn("match_year", F.year(F.col("date")))
)

In [11]:
prepared_df.printSchema()
prepared_df.select(
    "season",
    "date",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "total_goals",
    "goal_difference",
    "result",
    "match_year"
).show(10, truncate=False)

root
 |-- season: string (nullable = true)
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: integer (nullable = true)
 |-- away_goals: integer (nullable = true)
 |-- total_goals: integer (nullable = true)
 |-- goal_difference: integer (nullable = true)
 |-- result: string (nullable = false)
 |-- match_year: integer (nullable = true)

+------------------------+----------+---------------+---------------+----------+----------+-----------+---------------+--------+----------+
|season                  |date      |home_team      |away_team      |home_goals|away_goals|total_goals|goal_difference|result  |match_year|
+------------------------+----------+---------------+---------------+----------+----------+-----------+---------------+--------+----------+
|Championnat D1 2007/2008|2008-08-02|ASAC Concorde  |Nasr Sebkha    |2         |0         |2          |2              |home_win|2008      |
|Championnat D1 2007

### 1.5 — Null Value Analysis and Cleaning

We check for null values across all columns. The 154 null values in `date` and `match_year` come from malformed date strings that could not be parsed.

**Strategy**: Drop rows where any essential field (`season`, `date`, `home_team`, `away_team`, `home_goals`, `away_goals`) is null, as these are incomplete records that cannot be reliably used.

In [12]:
prepared_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in prepared_df.columns
]).show(truncate=False)

+------+----+---------+---------+----------+----------+-----------+---------------+------+----------+
|season|date|home_team|away_team|home_goals|away_goals|total_goals|goal_difference|result|match_year|
+------+----+---------+---------+----------+----------+-----------+---------------+------+----------+
|0     |154 |0        |0        |0         |0         |0          |0              |0     |154       |
+------+----+---------+---------+----------+----------+-----------+---------------+------+----------+



In [13]:
clean_df = prepared_df.dropna(
    subset=["season", "date", "home_team", "away_team", "home_goals", "away_goals"]
)

In [14]:
print("Rows before cleaning:", prepared_df.count())
print("Rows after cleaning:", clean_df.count())
print("Dropped rows:", prepared_df.count() - clean_df.count())

Rows before cleaning: 2744
Rows after cleaning: 2590
Dropped rows: 154


### 1.6 — Save Prepared Data in Parquet Format

We save the cleaned historical dataset in **Parquet format** for efficient reading by downstream tasks (streaming producer and ML training).

Parquet is a columnar format that provides efficient compression, schema preservation, and fast analytical queries.

In [15]:
output_path = "../outputs/processed_output"

clean_df.write.mode("overwrite").parquet(output_path)

## Part 2 - Current-Season Enrichment (Web Scraping)

In this part, we collect recent Mauritanian league match results from the web.

The objective is to build a dataset of current-season matches that will later be used as a test set for the machine learning model.

### 2.1 — Initial Exploration: Soccerway

We first attempt to scrape from **Soccerway** (`us.soccerway.com`), a popular football statistics site. However, the page uses heavy JavaScript rendering, making the HTML tables mostly empty when fetched with `requests`.

We therefore switch to **RSSSF** (Rec.Sport.Soccer Statistics Foundation), which provides match results in a simple, parseable text format.

In [16]:
from bs4 import BeautifulSoup
import re
import requests
url = "https://us.soccerway.com/mauritania/ligue-1-2024-2025/results/"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers, timeout=20)
html = response.text

print("Contains 'Latest Scores':", "Latest Scores" in html)
print("Contains 'Results':", "Results" in html)
print("Contains 'Round':", "Round" in html)
print("Contains '2024-2025':", "2024-2025" in html)
print("Contains 'MAURITANIA':", "MAURITANIA" in html)

# Try to find score-like patterns such as 1-0, 2:1, 0 - 0
score_patterns = re.findall(r"\b\d+\s*[-:]\s*\d+\b", html)
print("Number of score-like patterns found:", len(score_patterns))
print("First 20 score-like patterns:", score_patterns[:20])

soup = BeautifulSoup(html, "html.parser")
page_text = soup.get_text(" ", strip=True)

print(page_text[:2000])

Contains 'Latest Scores': True
Contains 'Results': True
Contains 'Round': True
Contains '2024-2025': True
Contains 'MAURITANIA': True
Number of score-like patterns found: 141
First 20 score-like patterns: ['2001-2026', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '1-2024', '084-1']
Soccerway Soccerway Favorites Matches Premier League LaLiga MLS Liga MX Serie A Bundesliga Champions League AD Mauritania Ligue 1 2024/2025 Summary Results Fixtures Standings Archive Latest Scores Show more games Scheduled Show more games 2024-2025 Mauritania Ligue 1 Pinned Leagues My Teams Albania Algeria Andorra Angola Antigua & Barbuda Argentina Armenia Aruba Australia Austria Bundesliga Azerbaijan Bahrain Bangladesh Barbados Belarus Belgium Jupiler Pro League Benin Bermuda Bhutan Bolivia Bosnia and Herzegovina Botswana Brazil Serie A Betano Bulgaria Burkina Faso Burundi Cambodia Ca

In [17]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

tables = soup.find_all("table")
print("Number of tables found:", len(tables))

for i, table in enumerate(tables[:10]):
    print("\n" + "=" * 80)
    print(f"TABLE {i}")
    print(table.get_text(" ", strip=True)[:1500])

Number of tables found: 2

TABLE 0
Show more games

TABLE 1
Show more games


### Data Extraction from RSSSF

The RSSSF website provides match results in a simple text format.

We extract the page content and convert it into plain text to identify match results line by line.

In [18]:
from bs4 import BeautifulSoup

rsssf_url = "https://www.rsssf.org/tablesm/maur2025.html"
response = requests.get(rsssf_url, timeout=20)

soup = BeautifulSoup(response.content, "html.parser")

page_text = soup.get_text("\n", strip=True)

print(page_text[:5000])

Mauritania 2024/25
Mauritania 2024/25
Championnat National de Premičre Division
Coupe Nationale
Super D2
Championnat National de Premičre Division
Final Table:

 1.Al-Hilal (Omdurman)       30  20  7  3  55-18  67  [*]
Champion d'honneur
2.FC Nouadhibou ASJN        30  17 10  3  37-13  61  [C]  Champions
 3.Chemal FC                 30  14  9  7  37-23  51
 4.Nouakchott King's         30  12 14  4  36-25  50
 5.AS Douanes                30  12 13  5  39-30  49
 6.Al-Merreikh (Omdurman)    30  12  8 10  36-28  44  [*]
 7.FC Tevragh-Zeďne          30  11 10  9  35-26  43
 8.AS Pompiers               30   9 11 10  30-29  38
 9.FC Inter Nouakchott       30   9 10 11  31-37  37
10.Kaédi FC                  30   9  9 12  37-49  36
11.ASC Gendrim               30   9  7 14  24-35  34
12.ASC SNIM                  30   8  8 14  24-28  32
13.FC Nzidane                30   6 12 12  30-41  30  [P]
14.FC Ksar                   30   7  8 15  19-31  29
------------------------------------------------

In [19]:
import re

lines = page_text.split("\n")

match_lines = []

clean_match_lines = []

for line in lines:
    line = line.strip()

    # keep only lines with score like 1-0
    if re.search(r"\d+\s*-\s*\d+", line):

        # remove lines with too many numbers (tables)
        if len(re.findall(r"\d+", line)) <= 4:
            clean_match_lines.append(line)

print("Clean match lines:", len(clean_match_lines))
print("\nFirst 20:\n")

for m in clean_match_lines[:20]:
    print(m)

Clean match lines: 269

First 20:

Kaédi FC                1-0 FC Inter Nouakchott
AS Garde                0-1 FC Ksar
Chemal FC               0-1 King's
FC Tevragh-Zeďne        3-0 AS Pompiers
ASC Touldé              3-2 ASC SNIM
ASC Gendrim             1-0 FC Nzidane
FC Nouadhibou           0-0 Al-Merreikh
Al-Hilal                1-1 AS Douanes
AS Pompiers             3-0 King's
FC Ksar                 1-2 Chemal FC
FC Tevragh-Zeďne        1-0 FC Nouadhibou
AS Douanes              1-1 ASC Touldé
ASC SNIM                1-0 AS Garde
FC Nzidane              0-0 Kaédi FC
FC Inter Nouakchott     1-4 Al-Hilal
Al-Merreikh             1-0 ASC Gendrim
Chemal FC               0-0 ASC SNIM
FC Nouadhibou           1-0 AS Pompiers
King's                  1-0 FC Ksar
AS Garde                0-2 AS Douanes


### Parsing Match Results

The extracted lines are filtered to keep only valid match results.

Each line is then transformed into structured fields:
- home_team
- away_team
- home_goals
- away_goals

This prepares the scraped data for integration with the historical dataset.

In [20]:
parsed_matches = []

for line in clean_match_lines:
    line = line.strip()

    # split only on the score part
    match = re.match(r"^(.*?)\s+(\d+)\s*-\s*(\d+)\s+(.*?)$", line)
    
    if match:
        home_team = match.group(1).strip()
        home_goals = int(match.group(2))
        away_goals = int(match.group(3))
        away_team = match.group(4).strip()

        parsed_matches.append({
            "home_team": home_team,
            "home_goals": home_goals,
            "away_goals": away_goals,
            "away_team": away_team
        })

print("Parsed matches:", len(parsed_matches))
print("\nFirst 10 parsed matches:\n")

for match in parsed_matches[:10]:
    print(match)

Parsed matches: 267

First 10 parsed matches:

{'home_team': 'Kaédi FC', 'home_goals': 1, 'away_goals': 0, 'away_team': 'FC Inter Nouakchott'}
{'home_team': 'AS Garde', 'home_goals': 0, 'away_goals': 1, 'away_team': 'FC Ksar'}
{'home_team': 'Chemal FC', 'home_goals': 0, 'away_goals': 1, 'away_team': "King's"}
{'home_team': 'FC Tevragh-Zeďne', 'home_goals': 3, 'away_goals': 0, 'away_team': 'AS Pompiers'}
{'home_team': 'ASC Touldé', 'home_goals': 3, 'away_goals': 2, 'away_team': 'ASC SNIM'}
{'home_team': 'ASC Gendrim', 'home_goals': 1, 'away_goals': 0, 'away_team': 'FC Nzidane'}
{'home_team': 'FC Nouadhibou', 'home_goals': 0, 'away_goals': 0, 'away_team': 'Al-Merreikh'}
{'home_team': 'Al-Hilal', 'home_goals': 1, 'away_goals': 1, 'away_team': 'AS Douanes'}
{'home_team': 'AS Pompiers', 'home_goals': 3, 'away_goals': 0, 'away_team': "King's"}
{'home_team': 'FC Ksar', 'home_goals': 1, 'away_goals': 2, 'away_team': 'Chemal FC'}


In [21]:
# --- Team Name Standardization ---
team_mapping = {
    "AS Garde": "Garde Nationale",
    "AS Douanes": "Douane",
    "FC Tevragh-Zeďne": "Tevragh-Zeina",
    "FC Tevragh-Zeïne": "Tevragh-Zeina",
    "Kaédi FC": "Kaedi",
    "Chemal FC": "Chemal",
    "ASC SNIM": "SNIM",
    "ASC Gendrim": "Gendrim",
    "King's": "Nouakchott King's",
    "FC Nouadhibou": "Nouadhibou",
    "ASAC Concorde": "ASAC Concorde",
    "FC Ksar": "Ksar",
    "AS Pompiers": "Pompiers",
    "ASC Touldé": "Toulde"
}

def standardize_team(name):
    name = name.replace("Zeďne", "Zeina").replace("Zeïne", "Zeina")
    name = name.replace("Kaédi", "Kaedi").replace("Touldé", "Toulde")
    for k, v in team_mapping.items():
        if k in name or name in k:
            return v
    return name

parsed_matches_clean = []
for m in parsed_matches:
    m['home_team'] = standardize_team(m['home_team'])
    m['away_team'] = standardize_team(m['away_team'])
    parsed_matches_clean.append(m)

scraped_df = spark.createDataFrame(parsed_matches_clean)

scraped_df.printSchema()
scraped_df.show(10, truncate=False)

root
 |-- away_goals: long (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: long (nullable = true)
 |-- home_team: string (nullable = true)

+----------+-------------------+----------+----------------+
|away_goals|away_team          |home_goals|home_team       |
+----------+-------------------+----------+----------------+
|0         |FC Inter Nouakchott|1         |Kaédi FC        |
|1         |FC Ksar            |0         |AS Garde        |
|1         |King's             |0         |Chemal FC       |
|0         |AS Pompiers        |3         |FC Tevragh-Zeďne|
|2         |ASC SNIM           |3         |ASC Touldé      |
|0         |FC Nzidane         |1         |ASC Gendrim     |
|0         |Al-Merreikh        |0         |FC Nouadhibou   |
|1         |AS Douanes         |1         |Al-Hilal        |
|0         |King's             |3         |AS Pompiers     |
|2         |Chemal FC          |1         |FC Ksar         |
+----------+-------------------+-------

In [22]:
scraped_df = (
    scraped_df
    .withColumn("season", F.lit("2024/2025"))
    .withColumn("date", F.lit(None).cast("date"))
)

scraped_df.printSchema()

root
 |-- away_goals: long (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: long (nullable = true)
 |-- home_team: string (nullable = true)
 |-- season: string (nullable = false)
 |-- date: date (nullable = true)



In [23]:
scraped_df = scraped_df.select(
    "season",
    "date",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals"
)

scraped_df.show(5, truncate=False)

+---------+----+----------------+-------------------+----------+----------+
|season   |date|home_team       |away_team          |home_goals|away_goals|
+---------+----+----------------+-------------------+----------+----------+
|2024/2025|NULL|Kaédi FC        |FC Inter Nouakchott|1         |0         |
|2024/2025|NULL|AS Garde        |FC Ksar            |0         |1         |
|2024/2025|NULL|Chemal FC       |King's             |0         |1         |
|2024/2025|NULL|FC Tevragh-Zeďne|AS Pompiers        |3         |0         |
|2024/2025|NULL|ASC Touldé      |ASC SNIM           |3         |2         |
+---------+----+----------------+-------------------+----------+----------+
only showing top 5 rows



In [24]:
scraped_df = (
    scraped_df
    .withColumn("total_goals", F.col("home_goals") + F.col("away_goals"))
    .withColumn("goal_difference", F.col("home_goals") - F.col("away_goals"))
    .withColumn(
        "result",
        F.when(F.col("home_goals") > F.col("away_goals"), "home_win")
         .when(F.col("home_goals") < F.col("away_goals"), "away_win")
         .otherwise("draw")
    )
)

In [26]:
scraped_df = (
    scraped_df
    .withColumn("match_year", F.lit(2025).cast("int"))
)

scraped_df.printSchema()

root
 |-- season: string (nullable = false)
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: long (nullable = true)
 |-- away_goals: long (nullable = true)
 |-- total_goals: long (nullable = true)
 |-- goal_difference: long (nullable = true)
 |-- result: string (nullable = false)
 |-- match_year: integer (nullable = false)



In [27]:
full_df = clean_df.unionByName(scraped_df)

print("Final dataset size:", full_df.count())
full_df.show(5, truncate=False)

Final dataset size: 2857
+------------------------+----------+---------------+-----------+----------+----------+-----------+---------------+--------+----------+
|season                  |date      |home_team      |away_team  |home_goals|away_goals|total_goals|goal_difference|result  |match_year|
+------------------------+----------+---------------+-----------+----------+----------+-----------+---------------+--------+----------+
|Championnat D1 2007/2008|2008-08-02|ASAC Concorde  |Nasr Sebkha|2         |0         |2          |2              |home_win|2008      |
|Championnat D1 2007/2008|2008-08-02|Police         |Tidjikja   |0         |1         |1          |-1             |away_win|2008      |
|Championnat D1 2007/2008|2008-08-01|Ksar           |SNIM       |1         |0         |1          |1              |home_win|2008      |
|Championnat D1 2007/2008|2008-08-01|Mauritel Mobile|Armee      |0         |0         |0          |0              |draw    |2008      |
|Championnat D1 2007/20

In [28]:
final_path = "../outputs/exported_dataset"

full_df.write.mode("overwrite").parquet(final_path)

## Summary

### What was produced

| Output | Path | Description |
|--------|------|-------------|
| Prepared historical data | `/workspace/data/mini_project/ready_data/` | 2590 cleaned matches (Parquet) |
| Final merged dataset | `/workspace/data/mini_project/exported_dataset/` | 2857 matches: historical + scraped (Parquet) |

### Data Quality Notes

- 154 rows dropped due to unparseable dates
- 267 scraped 2024/2025 matches from RSSSF (dates not available, set to NULL)
- Team names from RSSSF may differ slightly from historical data (e.g., encoding of special characters)

### Next Steps

1. **Part 3**: Run `producer.py` to send match events to Kafka
2. **Part 4**: Run `stream_job.py` for Spark Structured Streaming aggregations
3. **Part 5**: Open `train_model.ipynb` for ML model training and evaluation